# AEFS Third-Dataset Expansion — NASA MDP + ReLink

Runs the full AEFS pipeline on two additional benchmark pools:

**NASA MDP** (4 projects, 37 CK/Halstead/complexity metrics):
- PC3 (1077 files, 10% defective)
- PC4 (1458 files, 12% defective)
- cm1 (327 files, 10% defective)
- mw1 (253 files, 11% defective)

**ReLink** (3 projects, 26 Understand complexity metrics):
- Apache (194 files, 60% defective)
- Safe/OpenIntents (56 files, 39% defective)
- ZXing (399 files, 30% defective)

**Setup on Kaggle:**
1. Upload `AEFS_data.zip` as dataset (must include `nasa/` and `relink/` folders)
2. Add `HF_TOKEN` to Secrets
3. Run all cells

## 0. Setup

In [ ]:
!pip install -q lightgbm shap imbalanced-learn tqdm matplotlib seaborn python-dotenv huggingface_hub

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, time, math
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations
from scipy import stats
from scipy.stats import pointbiserialr
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif, RFE
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import matthews_corrcoef, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
import shap
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', font_scale=1.1)
RANDOM_STATE = 42
print('Setup complete.')

In [ ]:
# Data paths
RAW_NASA_DIR = Path('/kaggle/input/datasets/afrojahaqueshornaly/aefs-data/nasa')
RAW_RELINK_DIR = Path('/kaggle/input/datasets/afrojahaqueshornaly/aefs-data/relink')

# Auto-download if missing
import urllib.request
GITHUB_BASE = 'https://raw.githubusercontent.com/bharlow058/AEEEM-and-other-SDP-datasets/master/datasetcsv'

if not RAW_NASA_DIR.exists():
    print('Downloading NASA MDP from GitHub...')
    RAW_NASA_DIR = Path('/kaggle/working/nasa')
    RAW_NASA_DIR.mkdir(parents=True, exist_ok=True)
    for f in ['PC3.csv', 'PC4.csv', 'cm1.csv', 'mw1.csv']:
        urllib.request.urlretrieve(f'{GITHUB_BASE}/NASA/{f}', RAW_NASA_DIR / f)
        print(f'  Downloaded {f}')

if not RAW_RELINK_DIR.exists():
    print('Downloading ReLink from GitHub...')
    RAW_RELINK_DIR = Path('/kaggle/working/relink')
    RAW_RELINK_DIR.mkdir(parents=True, exist_ok=True)
    for f in ['apache.csv', 'safe.csv', 'zxing.csv']:
        urllib.request.urlretrieve(f'{GITHUB_BASE}/Relink/{f}', RAW_RELINK_DIR / f)
        print(f'  Downloaded {f}')

Path('/kaggle/working/results').mkdir(parents=True, exist_ok=True)
Path('/kaggle/working/figures').mkdir(parents=True, exist_ok=True)
RESULTS_DIR = '/kaggle/working/results'
FIG_DIR = '/kaggle/working/figures'

print(f'NASA files: {len(list(RAW_NASA_DIR.glob("*.csv")))}')
print(f'ReLink files: {len(list(RAW_RELINK_DIR.glob("*.csv")))}')

## 1. Data Loading

In [ ]:
def load_nasa():
    """Load NASA MDP: 4 projects (PC3, PC4, cm1, mw1), 37 features each.
    Target column varies: Defective (Y/N) or c (TRUE/FALSE).
    PC1 excluded — incompatible 22-column schema."""
    target_map = {'Y': 1, 'N': 0, 'TRUE': 1, 'FALSE': 0, 'true': 1, 'false': 0, 'yes': 1, 'no': 0, '1': 1, '0': 0}
    pool = {}
    for f in sorted(RAW_NASA_DIR.glob('*.csv')):
        name = f.stem.lower()
        df = pd.read_csv(f)
        # Find target column (Defective, c, or defects)
        target_col = None
        for col in ['Defective', 'c', 'defects']:
            if col in df.columns:
                target_col = col
                break
        if target_col is None:
            print(f'  Skipped {name}: no target column')
            continue
        df['defective'] = df[target_col].astype(str).str.strip().str.upper().map(target_map)
        df = df.drop(columns=[target_col])
        df = df.select_dtypes(include=[np.number])
        df = df.dropna().drop_duplicates()
        if len(df) >= 20 and df['defective'].nunique() == 2:
            pool[name] = df
        else:
            print(f'  Skipped {name}: {len(df)} rows, classes={df["defective"].nunique()}')
    return pool

def load_relink():
    """Load ReLink: 3 projects, 26 CK metrics, target=isDefective (buggy/clean)."""
    pool = {}
    for f in sorted(RAW_RELINK_DIR.glob('*.csv')):
        df = pd.read_csv(f)
        name = f.stem.lower()
        if 'isDefective' in df.columns:
            df['defective'] = df['isDefective'].map({'buggy': 1, 'clean': 0})
            df = df.drop(columns=['isDefective'])
        else:
            print(f'  Skipped {name}: no target column')
            continue
        df = df.select_dtypes(include=[np.number])
        df = df.dropna().drop_duplicates()
        if len(df) >= 20:
            pool[name] = df
    return pool

nasa_pool = load_nasa()
relink_pool = load_relink()

print(f'\nNASA MDP: {len(nasa_pool)} projects')
for name, df in nasa_pool.items():
    n_def = int(df['defective'].sum())
    print(f'  {name:10s}  rows={len(df):5d}  features={len(df.columns)-1:2d}  defective={n_def} ({100*n_def/len(df):.1f}%)')

print(f'\nReLink: {len(relink_pool)} projects')
for name, df in relink_pool.items():
    n_def = int(df['defective'].sum())
    print(f'  {name:10s}  rows={len(df):5d}  features={len(df.columns)-1:2d}  defective={n_def} ({100*n_def/len(df):.1f}%)')

## 2. Pipeline Functions

In [ ]:
LGBM_SFS_PARAMS = dict(n_estimators=60, max_depth=5, num_leaves=15, verbosity=-1, n_jobs=-1, random_state=RANDOM_STATE)
LGBM_FINAL_PARAMS = dict(n_estimators=300, max_depth=-1, num_leaves=31, learning_rate=0.05, verbosity=-1, n_jobs=-1, random_state=RANDOM_STATE)
RF_FINAL_PARAMS = dict(n_estimators=300, max_depth=None, n_jobs=-1, random_state=RANDOM_STATE)

In [ ]:
def rup_filter(df, variance_thr=1e-4, relevance_thr=0.02, redundancy_thr=0.90):
    cols = [c for c in df.columns if c != 'defective']
    X, y = df[cols], df['defective']
    vt = VarianceThreshold(threshold=variance_thr).fit(X)
    kept = [c for c, s in zip(cols, vt.get_support()) if s]
    relevant = [c for c in kept if abs(pointbiserialr(y, X[c])[0]) >= relevance_thr]
    final = list(relevant)
    for i in range(len(relevant)):
        for j in range(i + 1, len(relevant)):
            if relevant[i] in final and relevant[j] in final:
                if abs(X[relevant[i]].corr(X[relevant[j]])) >= redundancy_thr:
                    if abs(pointbiserialr(y, X[relevant[i]])[0]) < abs(pointbiserialr(y, X[relevant[j]])[0]):
                        final.remove(relevant[i])
                    else:
                        final.remove(relevant[j])
    if len(final) < 5:
        final = sorted(relevant, key=lambda c: abs(pointbiserialr(y, X[c])[0]), reverse=True)[:5]
    return final

def shap_refine(X, y, candidates, threshold_frac, min_features=5):
    if len(candidates) <= min_features:
        return candidates
    clf = LGBMClassifier(**LGBM_SFS_PARAMS).fit(X[candidates], y)
    sv = shap.TreeExplainer(clf).shap_values(X[candidates])
    if isinstance(sv, list): sv = sv[1]
    if sv.ndim == 3: sv = sv[:, :, 1]
    importance = pd.Series(np.abs(sv).mean(axis=0), index=candidates)
    total = importance.sum()
    if total == 0: return candidates
    keep = (importance / total)[(importance / total) >= threshold_frac].index.tolist()
    return keep if len(keep) >= min_features else importance.head(min_features).index.tolist()

def stability_check(X, y, project_labels, candidates, threshold, min_features=5):
    if len(candidates) <= min_features:
        return candidates, 1.0
    projects = list(project_labels.unique())
    project_top_sets = {}
    top_k = max(1, math.ceil(0.7 * len(candidates)))
    for proj in projects:
        mask = project_labels == proj
        Xp, yp = X.loc[mask, candidates], y.loc[mask]
        if len(Xp) < 20 or yp.value_counts().min() < 2: continue
        clf = LGBMClassifier(**LGBM_SFS_PARAMS).fit(Xp, yp)
        sv = shap.TreeExplainer(clf).shap_values(Xp)
        if isinstance(sv, list): sv = sv[1]
        if sv.ndim == 3: sv = sv[:, :, 1]
        ranked = pd.Series(np.abs(sv).mean(axis=0), index=candidates).sort_values(ascending=False)
        project_top_sets[proj] = set(ranked.head(top_k).index)
    if not project_top_sets: return candidates, 1.0
    n_eval = len(project_top_sets)
    presence = {f: sum(f in s for s in project_top_sets.values()) / n_eval for f in candidates}
    selected = [f for f in candidates if presence[f] >= threshold]
    if len(selected) < min_features:
        selected = sorted(candidates, key=lambda f: presence[f], reverse=True)[:min_features]
    proj_names = list(project_top_sets.keys())
    if len(proj_names) >= 2:
        jaccards = [_jaccard(project_top_sets[a], project_top_sets[b]) for a, b in combinations(proj_names, 2)]
        mean_jac = float(np.mean(jaccards))
    else:
        mean_jac = 1.0
    return selected, mean_jac

def _jaccard(a, b):
    return len(a & b) / len(a | b) if a or b else 1.0

In [ ]:
def lopo_folds(pool):
    projects = list(pool.keys())
    for target in projects:
        source_dfs = [pool[p].copy() for p in projects if p != target]
        train_df = pd.concat(source_dfs, ignore_index=True)
        test_df = pool[target].copy()
        y_train = train_df.pop('defective')
        y_test = test_df.pop('defective')
        scaler = MinMaxScaler()
        X_train = pd.DataFrame(scaler.fit_transform(train_df), columns=train_df.columns)
        X_test = pd.DataFrame(scaler.transform(test_df), columns=test_df.columns)
        proj_labels = pd.Series(np.concatenate([[p] * len(pool[p]) for p in projects if p != target]))
        yield {'target': target, 'X_train': X_train, 'y_train': y_train,
               'X_test': X_test, 'y_test': y_test, 'project_labels': proj_labels}

## 3. Full AEFS Pipeline + Baselines

In [ ]:
def run_stage4(pool, pool_name):
    models = {
        'lightgbm': ('lightgbm', LGBMClassifier(**LGBM_FINAL_PARAMS)),
        'random_forest': ('random_forest', RandomForestClassifier(**RF_FINAL_PARAMS)),
    }
    methods = ['AEFS', 'Ablation_no2D', 'AllFeatures', 'FilterMI', 'WrapperRFE']
    all_rows = []
    folds = list(lopo_folds(pool))
    print(f'[{pool_name}] {len(folds)} folds x {len(methods)} methods x {len(models)} models')

    for i, fold in enumerate(folds, 1):
        target = fold['target']
        X_tr, y_tr = fold['X_train'], fold['y_train']
        X_te, y_te = fold['X_test'], fold['y_test']
        proj_labels = fold['project_labels']

        train_w = X_tr.copy()
        train_w['defective'] = y_tr.values
        candidates = rup_filter(train_w)

        shap_feats = shap_refine(X_tr, y_tr, candidates, 0.02)
        aefs_feats, jac = stability_check(X_tr, y_tr, proj_labels, shap_feats, 0.30)

        k = max(1, min(len(aefs_feats), X_tr.shape[1]))
        sel_mi = SelectKBest(mutual_info_classif, k=k).fit(X_tr, y_tr)
        mi_feats = X_tr.columns[sel_mi.get_support()].tolist()[:k]
        rfe = RFE(LGBMClassifier(**LGBM_SFS_PARAMS), n_features_to_select=k).fit(X_tr, y_tr)
        rfe_feats = X_tr.columns[rfe.support_].tolist()

        method_feats = {
            'AEFS': aefs_feats, 'Ablation_no2D': shap_feats,
            'AllFeatures': list(X_tr.columns), 'FilterMI': mi_feats, 'WrapperRFE': rfe_feats,
        }
        method_jac = {'AEFS': jac}

        for method in methods:
            feats = method_feats[method]
            j = method_jac.get(method)
            for mname, (mlabel, mtemplate) in models.items():
                clf = mtemplate.__class__(**mtemplate.get_params())
                clf.fit(X_tr[feats], y_tr)
                proba = clf.predict_proba(X_te[feats])[:, 1]
                pred = (proba >= 0.5).astype(int)
                mcc = matthews_corrcoef(y_te, pred)
                f1v = f1_score(y_te, pred, zero_division=0)
                try: auc = roc_auc_score(y_te, proba)
                except: auc = float('nan')
                all_rows.append({
                    'pool': pool_name, 'target': target, 'method': method,
                    'model': mlabel, 'n_features': len(feats),
                    'mean_jaccard': round(j, 4) if j is not None else np.nan,
                    'test_mcc': round(mcc, 4), 'test_f1': round(f1v, 4),
                    'test_auc_roc': round(auc, 4),
                })
        print(f'  Fold {i}/{len(folds)}: {target} done')

    df = pd.DataFrame(all_rows)
    df.to_csv(f'{RESULTS_DIR}/{pool_name}_stage4_full_results.csv', index=False)
    print(f'Saved {len(df)} rows')
    return df

In [ ]:
nasa_results = run_stage4(nasa_pool, 'nasa')
relink_results = run_stage4(relink_pool, 'relink')

## 4. Results Summary

In [ ]:
def print_summary(df, ds_name):
    print('='*70)
    print(f'{ds_name} RESULTS')
    print('='*70)
    methods = ['AEFS', 'Ablation_no2D', 'AllFeatures', 'FilterMI', 'WrapperRFE']
    for model in ['lightgbm', 'random_forest']:
        print(f'\n--- {model.upper()} ---')
        for method in methods:
            s = df[(df['method'] == method) & (df['model'] == model)]
            j = s['mean_jaccard'].dropna().mean()
            jstr = f'{j:.3f}' if not np.isnan(j) else ' ---'
            print(f'  {method:20s} | n={s["n_features"].mean():5.1f} | J={jstr} | '
                  f'MCC={s["test_mcc"].mean():.3f}+/-{s["test_mcc"].std():.3f} | '
                  f'F1={s["test_f1"].mean():.3f} | AUC={s["test_auc_roc"].mean():.3f}')

print_summary(nasa_results, 'NASA MDP')
print_summary(relink_results, 'ReLink')

## 5. Statistical Tests

In [ ]:
from scipy import stats

def run_stats(df, ds_name):
    print('='*70)
    print(f'{ds_name} — WILCOXON + FRIEDMAN')
    print('='*70)
    for model in ['lightgbm', 'random_forest']:
        print(f'\n  {model.upper()}:')
        aefs = df[(df['method'] == 'AEFS') & (df['model'] == model)]['test_mcc'].values
        for bl in ['Ablation_no2D', 'AllFeatures', 'FilterMI', 'WrapperRFE']:
            blv = df[(df['method'] == bl) & (df['model'] == model)]['test_mcc'].values
            if len(aefs) == len(blv) and len(aefs) > 1:
                try:
                    stat, pval = stats.wilcoxon(aefs, blv, alternative='two-sided')
                    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
                    n1, n2 = len(aefs), len(blv)
                    delta = (sum(1 for a in aefs for b in blv if a > b) - sum(1 for a in aefs for b in blv if a < b)) / (n1 * n2)
                    eff = 'negl.' if abs(delta) < 0.147 else 'small' if abs(delta) < 0.33 else 'medium' if abs(delta) < 0.474 else 'large'
                    print(f'    vs {bl:20s}: W={stat:.1f}, p={pval:.4f} {sig}, d={delta:+.3f} ({eff})')
                except ValueError as e:
                    print(f'    vs {bl:20s}: {e}')
            else:
                print(f'    vs {bl:20s}: n={len(aefs)} (too few)')

        pivot = df[df['model'] == model].pivot_table(
            index='target', columns='method', values='test_mcc', aggfunc='first').dropna()
        avail = [m for m in ['AEFS','Ablation_no2D','AllFeatures','FilterMI','WrapperRFE'] if m in pivot.columns]
        mat = pivot[avail].values
        if mat.shape[0] > 2:
            chi2, pval = stats.friedmanchisquare(*[mat[:, i] for i in range(mat.shape[1])])
            sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
            print(f'    Friedman: chi2={chi2:.2f}, p={pval:.4f} {sig}')
        else:
            print(f'    Friedman: n={mat.shape[0]} (too few)')

run_stats(nasa_results, 'NASA MDP')
run_stats(relink_results, 'ReLink')

## 6. Figures

In [ ]:
def save_fig(name, fig):
    fig.savefig(os.path.join(FIG_DIR, f'{name}.pdf'), bbox_inches='tight')
    fig.savefig(os.path.join(FIG_DIR, f'{name}.png'), dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved: {name}')

colors = ['#e63946', '#f4a261', '#2a9d8f', '#264653', '#457b9d']
methods = ['AEFS', 'Ablation_no2D', 'AllFeatures', 'FilterMI', 'WrapperRFE']

In [ ]:
# NASA grouped bar
fig, ax = plt.subplots(figsize=(10, 5))
lgbm = nasa_results[nasa_results['model'] == 'lightgbm']
projects = lgbm['target'].unique()
x = np.arange(len(projects))
width = 0.15
for j, method in enumerate(methods):
    vals = [lgbm[(lgbm['target'] == proj) & (lgbm['method'] == method)]['test_mcc'].values[0]
            if len(lgbm[(lgbm['target'] == proj) & (lgbm['method'] == method)]) > 0 else 0
            for proj in projects]
    ax.bar(x + j*width, vals, width, label=method, color=colors[j], edgecolor='black', linewidth=0.5)
ax.set_xlabel('Target Project')
ax.set_ylabel('MCC')
ax.set_title('NASA MDP Per-Project MCC (LightGBM)')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(projects)
ax.legend(fontsize=8)
ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
save_fig('nasa_mcc_grouped_bar', fig)

In [ ]:
# ReLink grouped bar
fig, ax = plt.subplots(figsize=(8, 5))
lgbm = relink_results[relink_results['model'] == 'lightgbm']
projects = lgbm['target'].unique()
x = np.arange(len(projects))
for j, method in enumerate(methods):
    vals = [lgbm[(lgbm['target'] == proj) & (lgbm['method'] == method)]['test_mcc'].values[0]
            if len(lgbm[(lgbm['target'] == proj) & (lgbm['method'] == method)]) > 0 else 0
            for proj in projects]
    ax.bar(x + j*width, vals, width, label=method, color=colors[j], edgecolor='black', linewidth=0.5)
ax.set_xlabel('Target Project')
ax.set_ylabel('MCC')
ax.set_title('ReLink Per-Project MCC (LightGBM)')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(projects)
ax.legend(fontsize=8)
ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
save_fig('relink_mcc_grouped_bar', fig)

In [ ]:
# Feature count comparison (both datasets)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (res, name) in zip(axes, [(nasa_results, 'NASA MDP'), (relink_results, 'ReLink')]):
    lgbm = res[res['model'] == 'lightgbm']
    summary = lgbm.groupby('method')['n_features'].mean().reindex(methods)
    bars = ax.bar(summary.index, summary.values, color=colors, edgecolor='black', linewidth=0.5)
    for bar, val in zip(bars, summary.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_ylabel('Avg Features')
    ax.set_title(f'{name} Feature Count')
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
save_fig('feature_count_nasa_relink', fig)

## 7. Upload to HuggingFace

In [ ]:
from huggingface_hub import HfApi, login

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('Loaded HF_TOKEN from Kaggle Secrets')
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    HF_TOKEN = os.getenv('HF_TOKEN')
    if HF_TOKEN: print('Loaded HF_TOKEN from .env')

if not HF_TOKEN:
    raise RuntimeError('HF_TOKEN not found. Add via Kaggle Secrets.')

login(token=HF_TOKEN)
api = HfApi()
HF_REPO = 'MoshinAli/aefs-cpdp-results'
try:
    api.create_repo(HF_REPO, repo_type='dataset', private=True, exist_ok=True)
    print(f'Repo: https://huggingface.co/datasets/{HF_REPO}')
except Exception as e:
    print(f'Repo: {e}')

In [ ]:
upload_files = []
for f in os.listdir(RESULTS_DIR):
    if f.endswith('.csv') and ('nasa' in f.lower() or 'relink' in f.lower()):
        upload_files.append(os.path.join(RESULTS_DIR, f))
for f in os.listdir(FIG_DIR):
    if f.endswith(('.pdf', '.png')) and ('nasa' in f.lower() or 'relink' in f.lower()):
        upload_files.append(os.path.join(FIG_DIR, f))

print(f'Uploading {len(upload_files)} files to {HF_REPO}:\n')
for fp in upload_files:
    fname = os.path.basename(fp)
    ds = 'nasa' if 'nasa' in fname.lower() else 'relink'
    hf_path = f'expanded/{ds}/results/{fname}' if fname.endswith('.csv') else f'expanded/{ds}/figures/{fname}'
    try:
        api.upload_file(path_or_fileobj=fp, path_in_repo=hf_path, repo_id=HF_REPO, repo_type='dataset')
        print(f'  OK: {hf_path}')
    except Exception as e:
        print(f'  FAIL: {hf_path} -- {e}')
print(f'\nDone: https://huggingface.co/datasets/{HF_REPO}/tree/main')

## 8. Final Summary

In [ ]:
print('='*70)
print('EXPANSION — CROSS-DATASET AEFS SUMMARY (LightGBM)')
print('='*70)
print(f'  {"Dataset":15s} | {"Projects":>8s} | {"MCC":>8s} | {"F1":>8s} | {"AUC":>8s} | {"n_feat":>6s} | {"J":>6s}')
print('  ' + '-' * 75)
for ds_name, df in [('NASA MDP', nasa_results), ('ReLink', relink_results)]:
    s = df[(df['method'] == 'AEFS') & (df['model'] == 'lightgbm')]
    j = s['mean_jaccard'].dropna().mean()
    jstr = f'{j:.3f}' if not np.isnan(j) else ' ---'
    print(f'  {ds_name:15s} | {len(s):8d} | {s["test_mcc"].mean():8.3f} | {s["test_f1"].mean():8.3f} | {s["test_auc_roc"].mean():8.3f} | {s["n_features"].mean():6.1f} | {jstr}')

print(f'\nCombined with AEEEM (5 projects) and TeraPromise (68 projects):')
print(f'  Total: {len(nasa_pool) + len(relink_pool) + 5 + 68} projects across 4 datasets')